In [ ]:
import gzip
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier

from sktime.classification.kernel_based import RocketClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
with gzip.open("ts_normalized.pkl.gz", "rb") as f:
    ts = pickle.load(f)

for df in ts:
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)  # inutile
    if 'non-wear_flag' in df.columns:
        df.drop(columns=['non-wear_flag'], inplace=True)
    if 'weekday' in df.columns:
        df.drop(columns=['weekday'], inplace=True)
    if 'quarter' in df.columns:
        df.drop(columns=['quarter'], inplace=True)
    if 'relative_date_PCIAT' in df.columns:
        df.drop(columns=['relative_date_PCIAT'], inplace=True)

In [ ]:
ts_globale = pd.concat(ts, ignore_index=True)

# Opzionale: campionamento per velocizzare la generazione grafica in caso di dataset massivi.
# Se si desidera analizzare il 100% dei dati, lasciare frac=1.0.
df_plot = ts_globale.sample(frac=1.0, random_state=222)

# ==========================================
# 3. Visualizzazione tramite Istogrammi
# ==========================================
# Viene creata una griglia automatica in base al numero di colonne
df_plot.hist(bins=50, figsize=(12, 8), edgecolor='black',
             grid=False, alpha=0.75, color='steelblue')
plt.suptitle("Distribuzione Globale delle Variabili (Istogrammi)",
             y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:

X_list = []
y_list = []

for df in ts:
    # y è l'ultima colonna del DataFrame
    y_list.append(df.iloc[0, -1])

    # X contiene tutte le colonne tranne l'ultima
    df_features = df.iloc[:, :-1]
    X_list.append(df_features.values)

y = np.array(y_list)

array_norm = np.array(X_list)

X_sktime_norm = np.transpose(array_norm, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(
    X_sktime_norm,
    y,
    test_size=0.3,
    random_state=222,
    stratify=y
)

# 4. Verifica delle dimensioni
print("--- Dimensioni dei Dataset ---")
print(f"X_train shape: {X_train.shape} (Istanze, Variabili, Timepoints)")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

In [ ]:
# Minirocket

minirocket = RocketClassifier(
    rocket_transform="minirocket", random_state=222, n_jobs=-1)

# Fase di addestramento
# Assicurarsi che X_train sia nel formato supportato da sktime (es. array NumPy 3D [n_istanze, n_canali, n_timepoints])
minirocket.fit(X_train, y_train)

# Predizione sui dati di test
y_pred = minirocket.predict(X_test)

# Valutazione tramite metriche di performance
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.2f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.2f}")
print(f"F1: {f1_score(y_test, y_pred, average='weighted'):.2f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Generazione della matrice di confusione
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=minirocket.classes_)
disp.plot(cmap="Greens")
plt.grid(False)
plt.show()

In [ ]:
# 1. Creazione del Sottocampione (10% dei dati)
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train,
    y_train,
    train_size=0.10,
    random_state=222,
    stratify=y_train
)

# 2. Configurazione della Validazione Incrociata
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=222)

# 3. Griglie Separate per Euclidea e DTW
param_grid_euc = {
    'distance': ['euclidean'],
    'n_neighbors': [1, 3, 5, 7]
}

param_grid_dtw = {
    'distance': ['dtw'],
    'n_neighbors': [1, 3, 5, 7],
    'distance_params': [
        {"window": 0.05},
        {"window": 0.1},
        {"itakura_max_slope": 0.2},
        {"itakura_max_slope": 0.4}
    ]
}

# 4. Inizializzazione Classificatori Base (n_jobs=1 per evitare oversubscription)
knn_base_euc = KNeighborsTimeSeriesClassifier(n_jobs=1)
knn_base_dtw = KNeighborsTimeSeriesClassifier(n_jobs=1)

# Inizializzazione di due Grid Search separate
grid_search_euc = GridSearchCV(
    estimator=knn_base_euc, param_grid=param_grid_euc, cv=cv_strategy,
    scoring='accuracy', n_jobs=-1, verbose=1
)

grid_search_dtw = GridSearchCV(
    estimator=knn_base_dtw, param_grid=param_grid_dtw, cv=cv_strategy,
    scoring='accuracy', n_jobs=-1, verbose=1
)

# 5. Esecuzione delle Ricerche sul Sottocampione
print(f"--- Ottimizzazione KNN EUCLIDEO ({len(X_train_sub)} istanze) ---")
grid_search_euc.fit(X_train_sub, y_train_sub)
best_euc = grid_search_euc.best_estimator_
print(f"Migliori Parametri Euclidei: {grid_search_euc.best_params_}")
print(f"Accuracy CV (sottocampione): {grid_search_euc.best_score_:.4f}\n")

print(f"--- Ottimizzazione KNN DTW ({len(X_train_sub)} istanze) ---")
grid_search_dtw.fit(X_train_sub, y_train_sub)
best_dtw = grid_search_dtw.best_estimator_
print(f"Migliori Parametri DTW: {grid_search_dtw.best_params_}")
print(f"Accuracy CV (sottocampione): {grid_search_dtw.best_score_:.4f}\n")

# 6. Addestramento sull'Intero Dataset
print("Addestramento dei due modelli ottimali sull'intero dataset (3105 istanze)...")
best_euc.fit(X_train, y_train)
best_dtw.fit(X_train, y_train)

# 7. Valutazione sul Test Set
print("\nCalcolo predizioni sul Test Set (Euclideo)...")
y_pred_euc = best_euc.predict(X_test)

print("Calcolo predizioni sul Test Set (DTW)...")
y_pred_dtw = best_dtw.predict(X_test)

print("\n=========================================")
print(" RISULTATI FINALI SUL TEST SET (1332 istanze)")
print("=========================================")

print("\n>>> Modello KNN - EUCLIDEO <<<")
print(f"Accuracy: {accuracy_score(y_test, y_pred_euc):.4f}")
print(classification_report(y_test, y_pred_euc))

print("\n>>> Modello KNN - DTW <<<")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dtw):.4f}")
print(classification_report(y_test, y_pred_dtw))

# 8. Confronto Dettagliato Window vs Itakura (Analisi Interna CV)
risultati_cv_dtw = pd.DataFrame(grid_search_dtw.cv_results_)


def identifica_vincolo(params_dict):
    if isinstance(params_dict, dict):
        if 'window' in params_dict:
            return f"Window ({params_dict['window']})"
        elif 'itakura_max_slope' in params_dict:
            return f"Itakura ({params_dict['itakura_max_slope']})"
    return "Nessuno"


risultati_cv_dtw['Tipo Vincolo'] = risultati_cv_dtw['param_distance_params'].apply(
    identifica_vincolo)
colonne_display = ['param_n_neighbors', 'Tipo Vincolo',
                   'mean_test_score', 'std_test_score']
tabella_confronto_dtw = risultati_cv_dtw[colonne_display].sort_values(
    by='mean_test_score', ascending=False)

print("\n--- Confronto Dettagliato DTW (Sottocampione CV): Itakura vs Window ---")
print(tabella_confronto_dtw.to_string(index=False))

cm_euc = confusion_matrix(y_test, y_pred_euc)
cm_dtw = confusion_matrix(y_test, y_pred_dtw)

# 2. Configurazione dell'area di tracciamento (1 riga, 2 colonne)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Confronto Matrici di Confusione: KNN Euclideo vs KNN DTW", fontsize=16, y=1.05)

# 3. Plot della Matrice per il Modello Euclideo
disp_euc = ConfusionMatrixDisplay(
    confusion_matrix=cm_euc,
    display_labels=best_euc.classes_
)
disp_euc.plot(ax=axes[0], cmap='Blues', colorbar=True)
axes[0].set_title('Modello: KNN - Distanza Euclidea')
axes[0].grid(False)  # Rimuove la griglia sovrapposta ai colori

# 4. Plot della Matrice per il Modello DTW
disp_dtw = ConfusionMatrixDisplay(
    confusion_matrix=cm_dtw,
    display_labels=best_dtw.classes_
)
disp_dtw.plot(ax=axes[1], cmap='Greens', colorbar=True)
axes[1].set_title('Modello: KNN - Distanza DTW')
axes[1].grid(False)

# 5. Ottimizzazione degli spazi e visualizzazione
plt.tight_layout()
plt.show()